In [ ]:
import subprocess, sys

packages = [
    'torch', 'torchaudio',
    'asteroid',          # framework chứa Conv-TasNet + metrics
    'pesq',              # PESQ metric
    'pystoi',            # STOI metric
    'mir_eval',          # BSS eval (SDR)
    'soundfile',
    'pandas',
    'matplotlib',
    'tqdm'
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('Cài đặt xong')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torchaudio
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm import tqdm
from pathlib import Path


CHECKPOINT_PATH = '/kaggle/input/models/vanghh/hu/pytorch/default/1/last_checkpoint.pth'  

LIBRI3MIX_TEST_DIR = '/kaggle/input/datasets/habangchu/dev-test-libri3mix/librimix_16k/Libri3Mix/wav16k/min/test/'

SAMPLE_RATE = 16000      
N_SOURCES   = 3          
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_SAMPLES = None       

print(f'  Device: {DEVICE}')
print(f' Dataset dir: {LIBRI3MIX_TEST_DIR}')
print(f' Sample rate: {SAMPLE_RATE} Hz')


In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
print(ckpt.keys())


In [ ]:
from asteroid.models import ConvTasNet

def load_conv_tasnet(checkpoint_path, device='cpu', n_sources=3):
    print(f' Đang load checkpoint từ: {checkpoint_path}')
    ckpt = torch.load(checkpoint_path, map_location=device)
    epoch = ckpt.get('epoch', 'N/A')
    print(f'    Epoch đã huấn luyện: {epoch}')

    model = ConvTasNet(
        n_src=n_sources,
        sample_rate=SAMPLE_RATE,
        kernel_size=32,
        n_filters=512,
        stride=16,
        bn_chan=128,
        hid_chan=512,
        mask_act='relu',
        n_blocks=8,
        n_repeats=3,
        skip_chan=128
    )

    state_dict = ckpt['model_state_dict']
    new_sd = {k[len('module.'):] if k.startswith('module.') else k: v
              for k, v in state_dict.items()}

    model.load_state_dict(new_sd, strict=True)
    model = model.to(device)
    model.eval()
    total_params = sum(p.numel() for p in model.parameters())
    trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'    Tổng số tham số: {total_params:,}')
    print(f'    Tham số có thể huấn luyện: {trainable:,}')
    print(f'    Chạy trên: {device}')
    return model, epoch

model, trained_epoch = load_conv_tasnet(CHECKPOINT_PATH, device=DEVICE, n_sources=N_SOURCES)
print('\n Model sẵn sàng!')


In [ ]:
from torch.utils.data import Dataset, DataLoader


class Libri3MixTestDataset(Dataset):
    """
    Dataset cho Libri3Mix test split.

    Cấu trúc thư mục cần có:
        test/
          mix_clean/   (hoặc mix_both/)
          s1/
          s2/
          s3/

    Mỗi item trả về:
        mixture:  tensor (1, T)
        sources:  tensor (3, T)   – [s1, s2, s3]
        filename: str
    """
    def __init__(self, root_dir, sample_rate=16000, max_samples=None, mix_folder='mix_clean'):
        self.root    = Path(root_dir)
        self.sr      = sample_rate
        self.mix_dir = self.root / mix_folder
        self.s_dirs  = [self.root / f's{i+1}' for i in range(3)]

        if not self.mix_dir.exists():
            self.mix_dir = self.root / 'mix_clean'
        assert self.mix_dir.exists(), f"Không tìm thấy thư mục mix: {self.mix_dir}"

        self.files = sorted(self.mix_dir.glob('*.wav'))
        if max_samples:
            self.files = self.files[:max_samples]

        print(f' Dataset: {self.root}')
        print(f'   Mix dir: {self.mix_dir.name}')
        print(f'   Số file test: {len(self.files)}')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        mix, sr = torchaudio.load(fname)
        if sr != self.sr:
            mix = torchaudio.functional.resample(mix, sr, self.sr)

        sources = []
        for s_dir in self.s_dirs:
            s_path = s_dir / fname.name
            s, sr2 = torchaudio.load(s_path)
            if sr2 != self.sr:
                s = torchaudio.functional.resample(s, sr2, self.sr)
            sources.append(s[0])   # (T,)

        sources = torch.stack(sources)   # (3, T)
        return mix, sources, fname.stem


test_dataset = Libri3MixTestDataset(
    root_dir=LIBRI3MIX_TEST_DIR,
    sample_rate=SAMPLE_RATE,
    max_samples=MAX_SAMPLES
)

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

In [ ]:
from asteroid.metrics import get_metrics

all_results = []

print(f' Bắt đầu đánh giá trên {len(test_dataset)} file audio...')
print(f'   (device: {DEVICE} | SI-SDR, SDR, PESQ, STOI — PIT qua get_metrics)')

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating', unit='file'):
        mixture, sources, fname = batch
        fname = fname[0]

        mix_tensor = mixture.to(DEVICE)                          # (1, 1, T)
        estimates  = model(mix_tensor).squeeze(0).cpu().numpy()  # (3, T)

        targets = sources.squeeze(0).numpy()                     # (3, T)
        T = min(estimates.shape[1], targets.shape[1])
        estimates = estimates[:, :T]
        targets   = targets[:, :T]
        mix_np    = mixture.squeeze().numpy()[:T]                # (T,)

        try:
            m = get_metrics(
                mix=mix_np,
                clean=targets,
                estimate=estimates,
                sample_rate=SAMPLE_RATE,
                metrics_list=['si_sdr', 'sdr', 'pesq', 'stoi'],
                compute_permutation=True,
            )
        except Exception as e:
            print(f'  Bỏ qua file {fname}: {e}')
            continue

        all_results.append({
            'filename':   fname,
            'duration_s': round(T / SAMPLE_RATE, 2),
            'si_snr_mix': round(m['input_si_sdr'], 4),
            'si_snr':     round(m['si_sdr'], 4),
            'si_snri':    round(m['si_sdr'] - m['input_si_sdr'], 4),
            'sdr_mix':    round(m['input_sdr'], 4),
            'sdr':        round(m['sdr'], 4),
            'sdri':       round(m['sdr'] - m['input_sdr'], 4),
            'pesq':       round(m['pesq'], 4),
            'stoi':       round(m['stoi'], 4),
        })

df = pd.DataFrame(all_results)
print(f'\n Đánh giá hoàn tất! Số file: {len(df)}')


In [ ]:
metrics_cols = ['si_snr', 'si_snri', 'sdr', 'sdri', 'pesq', 'stoi']
summary = df[metrics_cols].agg(['mean', 'std', 'min', 'max']).round(4)

summary.columns = ['SI-SNR (dB)', 'SI-SNRi (dB)', 'SDR (dB)', 'SDRi (dB)', 'PESQ', 'STOI']
summary.index   = ['Mean', 'Std', 'Min', 'Max']

print('  KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH Conv-TasNet trên Libri3Mix test-clean')
print(f'  Checkpoint: last_checkpoint.pth     Epoch huấn luyện: {trained_epoch}')
print(f'  Số file test: {len(df)}     Sample rate: {SAMPLE_RATE} Hz')
print(summary.to_string())
print()
print(' Chỉ số chính (Mean):')
mean_row = df[metrics_cols].mean()
print(f'   SI-SNRi : {mean_row["si_snri"]:>8.4f} dB   (cải thiện so với hỗn hợp)')
print(f'   SDRi    : {mean_row["sdri"]:>8.4f} dB   (cải thiện so với hỗn hợp)')
print(f'   SI-SNR  : {mean_row["si_snr"]:>8.4f} dB')
print(f'   SDR     : {mean_row["sdr"]:>8.4f} dB')
print(f'   PESQ    : {mean_row["pesq"]:>8.4f}     (1–4.5, cao hơn = tốt hơn)')
print(f'   STOI    : {mean_row["stoi"]:>8.4f}     (0–1, cao hơn = tốt hơn)')

df.to_csv('libri3mix_test_results.csv', index=False)
summary.to_csv('libri3mix_test_summary.csv')
print('\n Đã lưu: libri3mix_test_results.csv  ,  libri3mix_test_summary.csv')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    f'Phân phối chỉ số đánh giá – Conv-TasNet trên Libri3Mix test-clean\n'
    f'(Epoch {trained_epoch}, n={len(df)} files)',
    fontsize=14, fontweight='bold', y=1.01
)

plot_configs = [
    ('si_snri',  'SI-SNRi (dB)',   '#2196F3', 'Chỉ số chính: cải thiện SI-SNR'),
    ('sdri',     'SDRi (dB)',      '#4CAF50', 'Cải thiện SDR'),
    ('si_snr',   'SI-SNR (dB)',    '#9C27B0', 'SI-SNR tuyệt đối'),
    ('sdr',      'SDR (dB)',       '#FF9800', 'SDR tuyệt đối'),
    ('pesq',     'PESQ',           '#F44336', 'Chất lượng cảm nhận (1–4.5)'),
    ('stoi',     'STOI',           '#00BCD4', 'Độ rõ ràng (0–1)'),
]

for ax, (col, label, color, title) in zip(axes.flat, plot_configs):
    data = df[col].dropna()
    mean_val = data.mean()
    std_val  = data.std()

    ax.hist(data, bins=30, color=color, alpha=0.75, edgecolor='white', linewidth=0.5)
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2,
               label=f'Mean = {mean_val:.2f}')
    ax.axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=1.5,
               label=f'±Std = {std_val:.2f}')
    ax.axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=1.5)

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Số file', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('metrics_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Đã lưu: metrics_distribution.png')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

categories = ['SI-SNR (dB)', 'SDR (dB)']
mix_vals   = [df['si_snr_mix'].mean(), df['sdr_mix'].mean()]
sep_vals   = [df['si_snr'].mean(),     df['sdr'].mean()]
impr_vals  = [df['si_snri'].mean(),    df['sdri'].mean()]

x = np.arange(len(categories))
width = 0.25

bars1 = ax.bar(x - width, mix_vals, width, label='Hỗn hợp (Mixture)', color='#9E9E9E', alpha=0.8)
bars2 = ax.bar(x,          sep_vals, width, label='Sau tách (Conv-TasNet)', color='#2196F3', alpha=0.8)
bars3 = ax.bar(x + width,  impr_vals, width, label='Cải thiện (Δ)', color='#4CAF50', alpha=0.8)

def autolabel(bars):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

autolabel(bars1)
autolabel(bars2)
autolabel(bars3)

ax.set_xlabel('Chỉ số', fontsize=12)
ax.set_ylabel('Giá trị (dB)', fontsize=12)
ax.set_title('So sánh Hỗn hợp vs Sau tách – Conv-TasNet / Libri3Mix test-clean', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='-')

plt.tight_layout()
plt.savefig('comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Đã lưu: comparison_bar.png')

In [ ]:
import torchaudio.transforms as T

mean_si_snri = df['si_snri'].mean()
idx_demo = (df['si_snri'] - mean_si_snri).abs().argmin()
demo_file = df.iloc[idx_demo]['filename']

print(f' File demo: {demo_file}')
print(f' SI-SNRi = {df.iloc[idx_demo]["si_snri"]:.4f} dB')

mix_path = Path(LIBRI3MIX_TEST_DIR) / 'mix_clean' / f'{demo_file}.wav'
if not mix_path.exists():
    mix_path = Path(LIBRI3MIX_TEST_DIR) / 'mix_both' / f'{demo_file}.wav'

mix_wav, _ = torchaudio.load(mix_path)
mix_tensor = mix_wav.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    est_wav = model(mix_tensor).squeeze(0).cpu()   # (3, T)

mix_np = mix_wav.squeeze().numpy()
est_np = est_wav.numpy()
for i in range(3):
    mmax = np.max(np.abs(est_np[i]))
    if mmax > 0: est_np[i] /= mmax

src_np = []
for i in range(3):
    s, _ = torchaudio.load(Path(LIBRI3MIX_TEST_DIR) / f's{i+1}' / f'{demo_file}.wav')
    src_np.append(s.squeeze().numpy())
T_min = min(est_np.shape[1], src_np[0].shape[0])
src_np = [s[:T_min] for s in src_np]
est_np = est_np[:, :T_min]
mix_np = mix_np[:T_min]
import itertools

def _si_sdr(est, ref, eps=1e-8):
    ref = ref - ref.mean()
    est = est - est.mean()
    alpha  = np.dot(est, ref) / (np.dot(ref, ref) + eps)
    target = alpha * ref
    noise  = est - target
    return 10 * np.log10((np.dot(target, target) + eps) / (np.dot(noise, noise) + eps))

src_arr = np.stack(src_np)                          # (3, T)
best_perm, best_score = None, -np.inf
for perm in itertools.permutations(range(len(src_np))):
    score = sum(_si_sdr(est_np[perm[j]], src_arr[j]) for j in range(len(src_np)))
    if score > best_score:
        best_score, best_perm = score, perm
est_np = est_np[list(best_perm)]                    
print(f'   Permutation áp dụng (Ŝ→S): {best_perm}')
t_axis = np.arange(T_min) / SAMPLE_RATE

fig, axes = plt.subplots(7, 1, figsize=(16, 20), sharex=True)
fig.suptitle(f'Trực quan hóa tách giọng nói\nFile: {demo_file}', fontsize=13, fontweight='bold')

axes[0].plot(t_axis, mix_np, color='#607D8B', linewidth=0.5)
axes[0].set_title('Hỗn hợp (Mixture)', fontweight='bold')
axes[0].set_ylabel('Biên độ')

colors_src = ['#E53935', '#43A047', '#1E88E5']
colors_est = ['#FF7043', '#66BB6A', '#42A5F5']

for i in range(3):
    ax_s = axes[1 + i]
    ax_e = axes[4 + i]

    ax_s.plot(t_axis, src_np[i], color=colors_src[i], linewidth=0.5)
    ax_s.set_title(f'Nguồn tham chiếu S{i+1}', fontweight='bold')
    ax_s.set_ylabel('Biên độ')

    ax_e.plot(t_axis, est_np[i], color=colors_est[i], linewidth=0.5, alpha=0.9)
    ax_e.set_title(f'Nguồn ước tính Ŝ{i+1}', fontweight='bold')
    ax_e.set_ylabel('Biên độ')

axes[-1].set_xlabel('Thời gian (giây)', fontsize=11)
for ax in axes:
    ax.grid(axis='both', alpha=0.2)

plt.tight_layout()
plt.savefig('waveform_visualization.png', dpi=130, bbox_inches='tight')
plt.show()
print(' Đã lưu: waveform_visualization.png')


In [ ]:
from IPython.display import Audio, display
import os

os.makedirs('audio_samples', exist_ok=True)

sf.write('audio_samples/mixture.wav', mix_np, SAMPLE_RATE)

for i in range(3):
    sf.write(f'audio_samples/reference_s{i+1}.wav', src_np[i], SAMPLE_RATE)
    sf.write(f'audio_samples/estimated_s{i+1}.wav', est_np[i], SAMPLE_RATE)

print(' Đã lưu audio mẫu vào thư mục audio_samples/')
print()

print(' Hỗn hợp (Mixture):')
display(Audio(mix_np, rate=SAMPLE_RATE))

for i in range(3):
    print(f'\n Nguồn tham chiếu S{i+1}:')
    display(Audio(src_np[i], rate=SAMPLE_RATE))
    print(f' Nguồn ước tính Ŝ{i+1}:')
    display(Audio(est_np[i], rate=SAMPLE_RATE))

In [ ]:
mean_vals = df[['si_snr_mix', 'si_snr', 'si_snri', 'sdr_mix', 'sdr', 'sdri', 'pesq', 'stoi']].mean()
std_vals  = df[['si_snr_mix', 'si_snr', 'si_snri', 'sdr_mix', 'sdr', 'sdri', 'pesq', 'stoi']].std()

print('        BẢNG KẾT QUẢ ĐÁNH GIÁ – Conv-TasNet / Libri3Mix test-clean      ')

print(f'  Epoch huấn luyện: {trained_epoch:<54}')
print(f'  Số lượng file test: {len(df):<52}')
print(f'  Sample rate: {SAMPLE_RATE} Hz                                                  ')
print('           Chỉ số                        Mean ± Std                     ')
rows = [
    ('SI-SNR hỗn hợp (dB)',    mean_vals['si_snr_mix'],  std_vals['si_snr_mix']),
    ('SI-SNR sau tách (dB)',   mean_vals['si_snr'],      std_vals['si_snr']),
    ('SI-SNRi (dB)  ',         mean_vals['si_snri'],     std_vals['si_snri']),
    ('SDR hỗn hợp (dB)',       mean_vals['sdr_mix'],     std_vals['sdr_mix']),
    ('SDR sau tách (dB)',      mean_vals['sdr'],         std_vals['sdr']),
    ('SDRi (dB)',              mean_vals['sdri'],        std_vals['sdri']),
    ('PESQ',                   mean_vals['pesq'],        std_vals['pesq']),
    ('STOI',                   mean_vals['stoi'],        std_vals['stoi']),
]
for name, m, s in rows:
    line = f'  {name:<36}    {m:>7.4f} ± {s:>6.4f}             '
    print(line)


In [ ]:
display_cols = ['filename', 'duration_s', 'si_snri', 'sdri', 'pesq', 'stoi']

print(' Top 5 file có SI-SNRi CAO NHẤT (tách tốt nhất):')
print(df.nlargest(5, 'si_snri')[display_cols].to_string(index=False))

print()
print(' Top 5 file có SI-SNRi THẤP NHẤT (tách khó nhất):')
print(df.nsmallest(5, 'si_snri')[display_cols].to_string(index=False))

print()
print(' Xem 10 file đầu tiên:')
print(df.head(10)[display_cols].to_string(index=False))